# Understanding PO in the capped setting

DP, January 2025

A straightforward adaptation of the standard fPO characterizations says that an allocation is Pareto if we can assign positive weights to agents such that every project funded strictly between 0 and its cap has total weighted valuation 1, all unfunded projects have <= 1, and all projects funded at their cap have >= 1.

But our definition of Lindahl equilibrium implicitly imposes == 1 for the latter type of project. Let's explore whether this is without loss of generality.

In [6]:
from gurobipy import GRB, Model, quicksum
import itertools
import random
from frozendict import frozendict
from enum import Enum

In [5]:
class Level(Enum):
    UNFUNDED = "Unfunded"
    PARTIALLY_FUNDED = "Partially Funded"
    FULLY_FUNDED = "Fully Funded"

In [7]:
def generate_all_mappings(projects):
    combinations = itertools.product(Level, repeat=len(projects))
    mappings = [frozendict(dict(zip(projects, combo))) for combo in combinations]
    return mappings

In [19]:
def inducible(v, mapping, restricted=False, verbose=False):
    N = v.keys()
    m = Model()
    m.params.OutputFlag = 0
    w = m.addVars(N, lb = 1)
    unfunded = [p for p in mapping if mapping[p] == Level.UNFUNDED]
    partially_funded = [p for p in mapping if mapping[p] == Level.PARTIALLY_FUNDED]
    fully_funded = [p for p in mapping if mapping[p] == Level.FULLY_FUNDED]
    total_score = {p : quicksum(w[i] * v[i][p] for i in N) for p in mapping}
    for x in fully_funded:
        for y in partially_funded:
            if restricted:
                m.addConstr(total_score[x] == total_score[y])
            else:
                m.addConstr(total_score[x] >= total_score[y])
    for x in partially_funded:
        for y in unfunded:
            m.addConstr(total_score[x] >= total_score[y])
    for x, y in itertools.combinations(unfunded, 2):
        m.addConstr(total_score[x] == total_score[y])
    if restricted:
        for x, y in itertools.combinations(fully_funded, 2):
            m.addConstr(total_score[x] == total_score[y])
    m.optimize()
    if verbose and m.status == GRB.OPTIMAL:
        print(f"w = {[w[i].x for i in N]}")
    return m.status == GRB.OPTIMAL

In [21]:
def inducible(v, mapping, restricted=False, verbose=False):
    N = v.keys()
    m = Model()
    m.params.OutputFlag = 0
    w = m.addVars(N)
    unfunded = [p for p in mapping if mapping[p] == Level.UNFUNDED]
    partially_funded = [p for p in mapping if mapping[p] == Level.PARTIALLY_FUNDED]
    fully_funded = [p for p in mapping if mapping[p] == Level.FULLY_FUNDED]
    total_score = {p : quicksum(w[i] * v[i][p] for i in N) for p in mapping}
    for x in fully_funded:
        if restricted:
            m.addConstr(total_score[x] == 1)
        else:
            m.addConstr(total_score[x] >= 1)
    for x in partially_funded:
        m.addConstr(total_score[x] == 1)
    for x in unfunded:
        m.addConstr(total_score[x] <= 1)
    m.optimize()
    if verbose and m.status == GRB.OPTIMAL:
        print(f"w = {[w[i].x for i in N]}")
    return m.status == GRB.OPTIMAL

In [29]:
projects = range(5)
mappings = generate_all_mappings(projects)
v = {}
for i in range(5):
    v[i] = {p : random.randint(1, 10) for p in projects}
for mapping in mappings:
    is_inducible = inducible(v, mapping)
    is_restricted_inducible = inducible(v, mapping, restricted=True)
    if is_inducible != is_restricted_inducible:
        print("Mapping: ", mapping)
        print("Valuations:")
        for i in v:
            print(i, list(v[i].values()))
        print("Inducible: ", is_inducible)
        inducible(v, mapping, restricted=False, verbose=True)
        print("Restricted Inducible: ", is_restricted_inducible)
        print()
        break

Mapping:  frozendict.frozendict({0: <Level.UNFUNDED: 'Unfunded'>, 1: <Level.UNFUNDED: 'Unfunded'>, 2: <Level.UNFUNDED: 'Unfunded'>, 3: <Level.PARTIALLY_FUNDED: 'Partially Funded'>, 4: <Level.FULLY_FUNDED: 'Fully Funded'>})
Valuations:
0 [10, 1, 2, 4, 10]
1 [9, 8, 6, 5, 8]
2 [9, 8, 3, 7, 10]
3 [1, 1, 6, 3, 9]
4 [8, 6, 6, 1, 6]
Inducible:  True
w = [0.012048192771084331, 0.0, 0.08433734939759037, 0.12048192771084337, 0.0]
Restricted Inducible:  False

